# Classifier Guidance 与 Energy Guidance：系统推导笔记

本笔记面向通用扩散模型理论，目标是严谨解释以下问题：

1. 分类器引导（classifier guidance）的数学原理与完整推导。
2. $\nabla_{x_t}\log p(c\mid x_t,t)$ 的来源与计算方式。
3. 能量函数替代分类器时，对应的概率模型到底是什么。
4. 什么时候它是 Boltzmann/Gibbs 分布，什么时候不是。
5. 为什么实践中常采用能量引导而不是显式分类器。

## 0. 记号与前向扩散

设数据变量为 $x_0\sim p_{\text{data}}(x)$，扩散时刻 $t\in[0,T]$。

以 VP-SDE 为例：
$$
\mathrm{d}x = -\frac{1}{2}\beta(t)x\,\mathrm{d}t + \sqrt{\beta(t)}\,\mathrm{d}W_t.
$$

其边缘分布可写为
$$
q(x_t\mid x_0)=\mathcal{N}(\alpha_t x_0,\sigma_t^2 I),
$$
其中
$$
\alpha_t = \exp\!\left(-\frac{1}{2}\int_0^t \beta(s)\,\mathrm{d}s\right),
\qquad
\sigma_t^2 = 1-\alpha_t^2.
$$

常用 score 定义：
$$
s_t(x_t) := \nabla_{x_t}\log p_t(x_t).
$$

我们希望采样条件分布 $p_t(x_t\mid c)$，其中 $c$ 是条件变量（类别、文本、约束、场景信息等）。

## 1. Classifier Guidance 的核心推导

从 Bayes 公式出发：
$$p(x_t | \text{guide} = c) = \frac{p(x_t) p(c | x_t)}{p(c)}$$
$$
\log p_t(x_t\mid c)=\log p_t(x_t)+\log p_t(c\mid x_t,t)-\log p_t(c).
$$
对 $x_t$ 求梯度：
$$
\nabla_{x_t}\log p_t(x_t\mid c)
=\nabla_{x_t}\log p_t(x_t)+\nabla_{x_t}\log p_t(c\mid x_t,t).
$$
$$\underbrace{\nabla_{x_t} \log p_t(x_t)}_{\text{修正后的方向}} \approx \underbrace{\nabla_{x_t} \log q_t(x_t)}_{\text{原始模型的输出}} + \underbrace{\gamma \nabla_{x_t} \log p(y | x_t)}_{\text{分类器/能量函数的梯度}}$$
这里的 $\gamma$ 是引导尺度（Scale），用来控制这只“手”的力量有多大。
因此条件 score 可分解为
$$
s_t^{(c)}(x_t)=s_t(x_t)+\nabla_{x_t}\log p_t(c\mid x_t,t).
$$

设网络给出无条件 score 近似 $s_\theta(x_t,t)$，并引入引导强度 $w\ge 0$，则
$$
\tilde s_t^{(c)}(x_t)=s_\theta(x_t,t)+w\,\nabla_{x_t}\log p_t(c\mid x_t,t).
$$

若模型使用噪声参数化，满足
$$
\epsilon_\theta(x_t,t)=-\sigma_t s_\theta(x_t,t),
$$
则有经典引导噪声形式：
$$
\epsilon_{\text{guided}}(x_t,t)=\epsilon_\theta(x_t,t)-w\,\sigma_t\,\nabla_{x_t}\log p_t(c\mid x_t,t).
$$

上式就是“分类器梯度如何进入扩散采样更新”的核心机制。

## 2. 不同参数化下的等价表达

### 2.1 $x_0$ 参数化
若网络输出 $\hat x_0(x_t,t)$，则
$$
\hat\epsilon_\theta(x_t,t)=\frac{x_t-\alpha_t\hat x_0(x_t,t)}{\sigma_t}.
$$
加入引导后：
$$
\hat\epsilon_{\text{guided}}=\hat\epsilon_\theta-w\sigma_t g_t,
\qquad g_t:=\nabla_{x_t}\log p_t(c\mid x_t,t).
$$
代回可得
$$
\hat x_{0,\text{guided}}
=\frac{x_t-\sigma_t\hat\epsilon_{\text{guided}}}{\alpha_t}
=\hat x_0 + \frac{w\sigma_t^2}{\alpha_t}g_t.
$$

### 2.2 score 参数化
若网络直接输出 score，直接替换为
$$
\tilde s_t^{(c)}=s_\theta+w g_t.
$$

### 2.3 解释
引导本质是把采样轨迹沿着“提高条件一致性”的方向推进，其步长由 $w$ 与噪声日程共同调制。

## 3. 能量函数替代分类器：它到底是不是 Boltzmann 分布

这是最容易混淆的点，结论分为“严格概率模型”和“工程近似”两层。

### 3.1 严格概率模型：是 Gibbs/Boltzmann 形式
若定义能量函数 $E_\phi(x_t,c,t)$，并显式归一化：
$$
p_\phi(c\mid x_t,t)
=\frac{\exp\big(-\beta E_\phi(x_t,c,t)\big)}{Z_\phi(x_t,t)},
\qquad
Z_\phi(x_t,t)=\sum_{c'}\exp\big(-\beta E_\phi(x_t,c',t)\big),
$$
则这就是标准 Gibbs/Boltzmann 分布（离散 $c$ 时是 softmax 形式）。

此时
$$
\nabla_{x_t}\log p_\phi(c\mid x_t,t)
=-\beta\nabla_{x_t}E_\phi(x_t,c,t)-\nabla_{x_t}\log Z_\phi(x_t,t).
$$
并且
$$
\nabla_{x_t}\log Z_\phi(x_t,t)
=\mathbb{E}_{c'\sim p_\phi(\cdot\mid x_t,t)}\left[-\beta\nabla_{x_t}E_\phi(x_t,c',t)\right].
$$

### 3.2 工程近似：不一定是完整 Boltzmann 概率
实践里常直接构造势函数 $\Phi(x_t,c,t)$（例如 reward、约束打分），并用
$$
g_t \approx \nabla_{x_t}\Phi(x_t,c,t)
$$
替代 $\nabla_{x_t}\log p(c\mid x_t,t)$。

这时若没有给出可积且有限的配分函数 $Z$，或显式忽略 $\nabla_{x_t}\log Z$，则它不再是严格归一化的条件概率梯度，而是“未归一化能量/势函数引导”。

因此：
1. 严格归一化时，它就是 Boltzmann/Gibbs。
2. 仅用单个 reward 梯度时，通常是近似引导场，不必对应某个完整概率分布。

## 4. $\nabla_{x_t}\log p(c\mid x_t,t)$ 在实现中如何计算

### 4.1 显式分类器（logits）
设分类器输出 logits $z_k(x_t,t)$，则
$$
\log p(c\mid x_t,t)=z_c(x_t,t)-\log\sum_j \exp\big(z_j(x_t,t)\big).
$$
于是
$$
\nabla_{x_t}\log p(c\mid x_t,t)
=\nabla_{x_t}z_c(x_t,t)-\sum_j p(j\mid x_t,t)\nabla_{x_t}z_j(x_t,t).
$$

### 4.2 自动求导实现（通用）
无论是分类器还是能量函数，工程上统一写成可微标量 $L(x_t,t,c)$，然后
$$
g_t = \nabla_{x_t} L(x_t,t,c).
$$

伪代码：
```python
x_in = x_t.detach().requires_grad_(True)
L = objective_fn(x_in, t, c)  # log p(c|x_t,t) 或其代理
g = torch.autograd.grad(L.sum(), x_in)[0]
```

关键点：
1. `requires_grad_(True)` 打开对输入的梯度。
2. `L.sum()` 把 batch 标量聚合，便于一次性求导。
3. 得到的 `g` 直接进入采样更新。

## 5. 输入到底应是 $x_t$ 还是 $x_0$

从严格定义看，引导项是
$$
\nabla_{x_t}\log p(c\mid x_t,t),
$$
因此自变量应当是当前时刻状态 $x_t$。

但很多实现会构造一个“低噪声代理”
$$
x_{\text{proxy}} = x_t + \operatorname{stopgrad}(\hat x_0(x_t,t)-x_t),
$$
在 $x_{\text{proxy}}$ 上评估几何/物理 reward，再把梯度传回 $x_t$。

链式法则给出
$$
\nabla_{x_t}\Phi(x_{\text{proxy}})
=\left(\frac{\partial x_{\text{proxy}}}{\partial x_t}\right)^\top
\nabla_{x_{\text{proxy}}}\Phi.
$$
当 stop-gradient 使雅可比近似恒等时，可得
$$
\nabla_{x_t}\Phi(x_{\text{proxy}})\approx \nabla_{x_{\text{proxy}}}\Phi.
$$

这是一种稳定性与语义可解释性的折中：
1. 数值上像在“更干净的状态”上算约束。
2. 变量上仍可通过链式法则回传到 $x_t$。

## 6. 为什么常用能量引导而不是显式分类器

### 6.1 设计动机
1. 不需要额外训练一个高质量噪声条件分类器。
2. 可以直接注入可微先验（物理约束、安全距离、几何规则、任务代价）。
3. 易于模块化组合：多个能量项线性加权即可。
4. 在工程中更容易做温度、时间窗、尺度调度。

### 6.2 代价与风险
1. 概率语义可能不严格：若忽略 $\nabla\log Z$，则不对应精确后验梯度。
2. 引导尺度敏感：$w$ 过大可能导致采样偏移或不稳定。
3. 能量形状决定优化地形，可能引入局部极值与不平滑梯度。

### 6.3 常见缓解策略
1. 时间调度：仅在特定噪声区间启用或增强引导。
2. 梯度裁剪/归一化：控制异常大梯度。
3. 平滑与低通处理：减小轨迹抖动。
4. 多项能量权重退火：先可行性，后细节。

## 7. 结论与快速判断

1. **严格 classifier guidance**：
$$
g_t=\nabla_{x_t}\log p(c\mid x_t,t),
$$
输入变量是 $x_t$，这是理论定义。

2. **能量替代分类器**：
若定义
$$
p(c\mid x_t,t)\propto \exp\big(-\beta E(x_t,c,t)\big),
$$
并保留归一化项，则属于 Gibbs/Boltzmann 框架；
若只用单项 reward/energy 梯度做引导而未显式处理配分函数，则通常是未归一化势函数近似，不必对应严格概率分布。

3. **为什么这样设计**：
它牺牲部分严格概率一致性，换取更强的可控性、可解释约束注入能力和工程可落地性。

4. **实践建议**：
当任务强调概率校准时，优先显式分类器或完整 EBM 归一化训练；
当任务强调可行性与规则控制时，能量引导往往更高效。